In [1]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "shuttle.csv"
processed_file = ROOT / "preprocessed_data" / "shuttle.csv"
artifact_dir = ROOT / "artifacts"

print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=MEDGAN(
        ae_pretrain_epochs=10,
        gan_epochs=10
    )
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="shuttle",
    artifact_store=store,
    model_name="medgan",
)

print(results)

Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\shuttle.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\shuttle.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\shuttle.csv
Loaded data with shape: (58000, 10)


INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (46400, 9)
INFO:katabatic.models.medgan.models:Categorical columns: ['class']
INFO:katabatic.models.medgan.models:Continuous columns: ['time', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8']
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 54.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]
INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 10 epochs...


Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved dataset artifact under datasets/shuttle/split-20260808-010559


INFO:katabatic.models.medgan.models:Epoch 1/10: AE Loss = 0.583786
INFO:katabatic.models.medgan.models:Epoch 10/10: AE Loss = 0.506284
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: D Loss = 0.963346, G Loss = 1.160747
INFO:katabatic.models.medgan.models:Epoch 10/10: D Loss = 0.063848, G Loss = 13.224518
INFO:katabatic.models.medgan.models:
Generating 46400 synthetic samples...
INFO:katabatic.models.medgan.models:Adding one existing training sample for missing target classes...
INFO:katabatic.models.medgan.models:
Synthetic data saved to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\models\medgan_shuttle_train-20260808-010559\synthetic
INFO:katabatic.models.medgan.models:Training complete!


{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='shuttle', dataset_version='split-20260808-010559'), 'model_ref': ModelRef(model_name='medgan', dataset_name='shuttle', dataset_version='split-20260808-010559', train_run_id='train-20260808-010559'), 'evaluation_refs': []}


In [2]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "shuttle"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = []

continuous_cols = [
    "time",
    "a1",
    "a2",
    "a3",
    "a4",
    "a5",
    "a6",
    "a7",
    "a8",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\shuttle\split-20260808-010559
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
       time        a1         a2        a3         a4        a5         a6  \
0  3.604638  8.387594   4.250953  3.069042   3.051358  4.244540   7.307993   
1  3.015673  5.924513  12.201419  3.518312  14.296370  3.774763  10.416147   
2  2.604345  5.917260  11.228394  3.443974  13.748208  3.545332  10.490747   
3  3.507116  5.958476  12.781124  3.610447  15.135489  3.942447  10.303289   
4  4.283951  8.669133   4.813642  3.064253   3.022200  4.272559   7.296647   

          a7         a8  class  
0  40.044605  28.302122      2  
1   9.516511   1.785996      1  
2  11.186375   2.012346      1  
3   7.496716   1.480987      1  
4  41.235252  28.798471      2  

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.8041

Continuous Wasserstein (

c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\O


=== Utility Evaluation ===
Overall utility score: 0.6910

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.6881       0.9548       0.2667
LR           f1         0.6996       0.9539       0.2543
DT           accuracy   0.6630       0.9995       0.3365
DT           f1         0.6747       0.9995       0.3248
RF           accuracy   0.6968       0.9996       0.3028
RF           f1         0.7004       0.9995       0.2991
LinearSVM    accuracy   0.5517       0.9382       0.3865
LinearSVM    f1         0.6392       0.9343       0.2951
MLP          accuracy   0.6805       0.9993       0.3188
MLP          f1         0.6937       0.9993       0.3056

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.7226

Bin Coverage (10 bins, % bins hit by synth)
  time                           60.0%
  a1                             30.0%
  a2                             50.0